# CNC 论文代码 — Jupyter 运行入口

论文代码已整理为**三层**，本 Notebook 按层逐步运行，你可以单独执行某一层或某一张图。

- **第 1 层 · 主代码** `core/cnc.py`：CNC 核心算法（被第 2 层调用，本身不直接运行）
- **第 2 层 · 算法比较 + 数据生成** `comparison/*`：调用主代码与 7 个基线对比，输出 `results/*.json`
- **第 3 层 · 根据数据生成图形** `visualization/*`：读取 `results/*.json`，输出 `figures/paper_figures/*`

> 请在 `CNC_code_organized/` 目录下用 Jupyter 打开本 Notebook（保证工作目录即项目根）。

In [1]:
import os, sys
# 把三个分类目录加入 sys.path，使彼此可以互相 import（无需关心当前工作目录）
PROJECT_ROOT = os.getcwd()
for sub in ('core', 'comparison', 'visualization'):
    p = os.path.join(PROJECT_ROOT, sub)
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)
# CPU is the reproducible reference backend; GPU is opt-in via cnc.set_device('gpu').
import cnc
cnc.set_device('cpu')
print('Compute backend =', cnc.backend_summary())
print('PROJECT_ROOT =', PROJECT_ROOT)
print('已加入 sys.path 的分类目录:',
      [s for s in ('core', 'comparison', 'visualization') if os.path.isdir(os.path.join(PROJECT_ROOT, s))])

Compute backend = CPU (SciPy eigensolver, scikit-learn neighbors and KMeans)
PROJECT_ROOT = /20140171/高云1016
已加入 sys.path 的分类目录: ['core', 'comparison', 'visualization']


## 第 1 层单独演示（可选）

直接用主代码 `cnc.cnc_cluster` 在自定义数据上跑一次，验证算法可用。

In [2]:
import numpy as np
import cnc
from sklearn.datasets import make_blobs
from sklearn.metrics import adjusted_rand_score

X, y = make_blobs(n_samples=600, centers=4, random_state=0, cluster_std=1.0)
out = cnc.cnc_cluster(X, k=20, Kmax=8, B=20, seed=0)   # 仅调用主代码
ari = adjusted_rand_score(y, out['labels'])
print('自动选择的 K =', out['K'], '| ARI =', round(float(ari), 4))

自动选择的 K = 4 | ARI = 0.8325


## 第 2 层：算法比较 + 数据生成

运行全部对比实验，把结果写入 `results/`。

In [3]:
import time
t0 = time.time()

import experiments
checks = experiments.run_all(include_extended=True)   # 主 benchmark / k 敏感 / rank 消融 / 多种子 / 运行时间 / 高光谱 / Indian Pines / K 选择 / 协议消融 / 审稿答复

# 额外审稿人补充实验
import additional_selectors, cross_affinity_validation
additional_selectors.main()
cross_affinity_validation.main()
cross_affinity_validation.margin_sensitivity()

print('第 2 层完成，耗时 %.0f s' % (time.time() - t0))

[experiments] [run] benchmark
[experiments] benchmark (synthetic + Gauss20D + Digits)


/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(


[experiments] Moons        CNC K=2 ARI=1.00


/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(


[experiments] Circles      CNC K=2 ARI=1.00
[experiments] Blobs        CNC K=4 ARI=0.83
[experiments] Aniso        CNC K=3 ARI=0.62
[experiments] VariedBlobs  CNC K=4 ARI=0.98


/usr/local/lib/python3.12/dist-packages/sklearn/cluster/_affinity_propagation.py:140: ConvergenceWarning: Affinity propagation did not converge, this model may return degenerate cluster centers and labels.
  warnings.warn(


[experiments] saved -> results/benchmark_results.json
[experiments] [run] k-sensitivity
[experiments] k-sensitivity
[experiments] saved -> results/k_sensitivity.json
[experiments] [run] rank-ablation
[experiments] rank-exponent ablation
[experiments] saved -> results/rank_ablation.json
[experiments] [run] multiseed
[experiments] multi-seed robustness
[experiments] saved -> results/multiseed_benchmark.json
[experiments] [run] runtime
[experiments] runtime scaling
[experiments] saved -> results/runtime_scaling.json
[experiments] [run] hsi
[experiments] hyperspectral (Asian Games Village)
[experiments] saved -> results/ayc_paper_metrics.json
[experiments] saved -> results/ayc_kstab_ext12.json
[experiments] saved -> results/ayc_kstab_ext.json
[experiments] saved -> results/ayc_kstab_ext24.json
[experiments] K=20
[experiments] [run] ayc-1se
[experiments] AYC 1-SE stability stats
[experiments] saved -> results/ayc_stability_stats.json
[experiments] [run] indian-pines
[experiments] Indian Pin

## 第 3 层：根据数据生成图形

读取 `results/`，绘制并导出 `figures/paper_figures/*`（PDF + 300dpi PNG）。

In [4]:
import os, time
t0 = time.time()

import experiments
experiments.render_all_figures()        # visualization/plotting.py 的 render_* 系列

import make_figures
make_figures.make_fig1()
make_figures.make_fig2()
make_figures.make_fig3()
make_figures.make_fig4()

# Fig5 标签文件名使用 results/ayc_paper_metrics.json 中的 selected K。
FIGS = os.path.join(PROJECT_ROOT, 'figures', 'paper_figures')
import json
with open(os.path.join(PROJECT_ROOT, 'results', 'ayc_paper_metrics.json'), encoding='utf-8') as f:
    AYC_K = int(json.load(f)['K'])
ayc_file = os.path.join(FIGS, f'ayc_cnc_labels_K{AYC_K}.npy')
if not os.path.exists(ayc_file):
    import regen_ayc
if os.path.exists(ayc_file):
    make_figures.make_fig5()

print('第 3 层完成，耗时 %.0f s' % (time.time() - t0))

[experiments] rendering figures


/20140171/高云1016/visualization/plotting.py:294: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


[experiments] all figures rendered
[Fig1] framework saved
[Fig2] affinity-vs-knn saved
[Fig3] refinement saved
[Fig4] benchmark saved
[regen] loading AYC (PCA->6) ...
[regen] X shape (40000, 6)
[regen] building resonance affinity ...
[regen] n_edges 5459876
[regen] spectral partition K=20 ...
[regen] KMeans reference K=20 ...
[regen] saved labels. unique cnc: 20 unique km: 20
[regen] side 200


/20140171/高云1016/visualization/make_figures.py:383: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout(rect=[0.0, 0.06, 1.0, 1.0])


[Fig5] hyperspectral saved
第 3 层完成，耗时 201 s


## 查看产物


In [5]:
import glob, os
print('数据文件 (results/):')
for f in sorted(glob.glob(os.path.join(PROJECT_ROOT, 'results', '*.json'))):
    print('  ', os.path.basename(f))
print('\n图形文件 (figures/paper_figures/):')
for f in sorted(glob.glob(os.path.join(PROJECT_ROOT, 'figures', 'paper_figures', '*'))):
    print('  ', os.path.basename(f))

数据文件 (results/):
   additional_selectors.json
   ayc_kstab_ext.json
   ayc_kstab_ext12.json
   ayc_kstab_ext24.json
   ayc_paper_metrics.json
   ayc_stability_stats.json
   benchmark_results.json
   cross_affinity_validation.json
   embedding_protocol_ablation.json
   embedding_protocol_multiseed.json
   indian_pines_benchmark.json
   k_sensitivity.json
   kselection_baselines.json
   margin_sensitivity_expanded.json
   multiseed_benchmark.json
   rank_ablation.json
   reviewer_response_checks.json
   runtime_scaling.json

图形文件 (figures/paper_figures/):
   Fig1.pdf
   Fig1.png
   Fig2.pdf
   Fig2.png
   Fig3.pdf
   Fig3.png
   Fig4.pdf
   Fig4.png
   Fig5.pdf
   Fig5.png
   ayc_cnc_labels_K20.npy
   ayc_compare.png
   ayc_compare_clean.png
   ayc_kmeans_labels_K20.npy
   ayc_kstab.pdf
   ayc_kstab.png
   ayc_spectra.pdf
   ayc_spectra_paper.pdf
   ayc_spectra_paper.png
   cnc_affinity.pdf
   cnc_idea.pdf
   cnc_rank_ablation.pdf
   cnc_synthetic.pdf
   cnc_synthetic.png
   fig_baseline

## 提示：单独重跑某一块

- 只重画某一张图：`make_figures.make_fig4()`
- 只重跑某个实验：`experiments.run_benchmark()` 然后 `experiments.render_all_figures()`
- 只重新出图（数据已存在）：直接运行上面的「第 3 层」单元即可。